# Lab 9.1 &mdash; Deploy the FrontDesk Service

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 45 min &nbsp;|&nbsp; **Day 3 &middot; Module 9 &mdash; Deployment &amp; AgentOps**

### What you'll do
- Read what your namespace allows, and what it can reach
- Make the five choices those limits force &mdash; including the one that has already broken this app once
- Lint the whole manifest set offline, before anything is scheduled
- Apply it, wait for the rollout, and get a grounded answer out of the running service
- Read your own trace back out of Tempo &mdash; the only evidence telemetry works

> **How this lab works.** You write real FastAPI, Pydantic, LangChain and Kubernetes-manifest
> code. Fill every `BLANK`, then run the **Self-check** cell under each section &mdash; those
> assert on the *objects you built* (a route table, a request contract, a compiled tool, a
> manifest dict), so they are deterministic. **No graded cell needs a cluster, a running server
> or a model.** Cells marked **Run it for real** put your code in front of the sandbox model,
> your own namespace or the tracing backend; if any of those is unreachable they print how to
> fix it instead of crashing. The score line is feedback, not a grade.

> **Everything before this ran in a notebook.** This lab puts a multi-agent service
> on a cluster, in your own namespace, on your own hostname, reachable from a
> browser. You are not asked to type the manifests out: they are written for you,
> and what you supply are the five decisions inside them.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap
from typing import Any, Callable, Optional

WORK = os.path.join("/tmp", "awmas-lab-9-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model can reason before it answers, and the reasoning is billed as completion
# tokens. Off is the default here because a deployment lab makes a lot of small calls.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL, api_key=LLM_API_KEY,
                          temperature=temperature, extra_body=NO_THINK)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- your own namespace --------------------------------------------------
# You deploy into your own namespace, published at your own host. Both are injected into
# the sandbox, so nothing here is hardcoded and nothing here needs them to be set.
#
# Read ONLY from APP_NAMESPACE, never derived from the hostname. A cell below runs
# kubectl against whatever this says, and a namespace guessed from a machine name is
# the wrong thing to point kubectl at.
APP_NS   = os.environ.get("APP_NAMESPACE", "")
APP_HOST = os.environ.get("APP_HOST", "")

print("work dir :", WORK)
print("model    :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")
print("namespace:", APP_NS or "(unknown -- no graded cell needs it)")

# ---- this lab clones a repo, so it works under ~/work, not /tmp -------------
# /tmp does not survive an OOMKill and a restart mid-lab would take the clone.
HOME_WORK = os.path.expanduser("~/work")
if APP_NS:                       # always present in a sandbox; absent on a laptop
    os.makedirs(HOME_WORK, exist_ok=True)
print("workdir   :", HOME_WORK)

## What you are deploying, and where

**FrontDesk AI** &mdash; a FastAPI service wrapping the LangGraph support desk you have been
studying all week: supervisor &rarr; RAG &rarr; a domain worker with tools &rarr; escalation
&rarr; QA gate. The image is already built and published, so there is nothing to compile here.

You are putting it in **your own namespace**, on **your own hostname**, and a browser will
reach it. That is a different job from running a notebook, and most of it is decided before
`kubectl` is involved.

### Your namespace is not an unlimited machine

Read these off your own quota any time with
`kubectl -n $APP_NAMESPACE describe quota`. Three of them force a decision below.

| Limit | Value | Why it matters |
|---|---|---|
| `limits.memory` | **2Gi &mdash; for the whole namespace** | your app's own limit is 1Gi, so it is half the budget |
| `services.nodeports` | **0** | not "a few". Zero |
| `services.loadbalancers` | **0** | so there is exactly one way in |
| `services` | 4 | |
| `persistentvolumeclaims` | 4 | |
| `pods` | 8 | |
| `requests.storage` | 5Gi | |

### And it can only reach four things

Its NetworkPolicy is `participant-egress`. This is **not** the sandbox your notebook runs
in &mdash; that one does have general internet access. Yours does not.

| Allowed out | For |
|---|---|
| `kube-system` :53/UDP | DNS |
| `llm-serving` any port | the LiteLLM gateway &mdash; how the app reaches a model |
| `ingress-nginx` pods | so your Ingress can reach back in |
| `tempo` :4317 &middot; `langfuse` :3000 | telemetry |

Nothing else on the internet. Not PyPI, not Hugging Face, not a model vendor.

### Three Secrets, and one of them is yours to supply

You never copy a value out of any of these. They arrive in the pod by reference.

| Secret | Where it comes from | What is in it |
|---|---|---|
| `<your-namespace>-llm` | published for you | the gateway URL, the model name, and your own daily-capped key |
| `frontdeskai-secret` | created by the deploy script | `SECRET_KEY` and the first-login password |
| `frontdeskai-langfuse` | **created by the deploy script from your own `.env`** | your Langfuse public key, secret key and region host |

### Before you deploy: clone the app, and bring your own Langfuse keys

Two things to do first, both in a terminal.

**1. Clone the app.** You have been reading this code all week; now you need it on disk, because
its own deploy script is what puts it on the cluster.

```
git clone https://github.com/brainupgrade-in/aiagentic-comp-frontdeskai.git ~/work/frontdeskai
```

**2. Give it your own Langfuse project.** Tracing is per participant, so nobody can read anyone
else's prompts. Sign in at Langfuse, create a project, take an API key pair, then:

```
cd ~/work/frontdeskai
cp .env.example .env        # then fill in the three LANGFUSE_ values
```

That is the **same `.env` the app reads with `load_dotenv()`** when you run it locally, so one
file configures both paths. It is gitignored, so your keys cannot be committed.

⚠️ **Two ways this fails silently, and neither raises an error.** The app reads `LANGFUSE_HOST`;
the SDK docs call the same thing `LANGFUSE_BASE_URL`, so the deploy script accepts either and
translates &mdash; but a third spelling disables tracing with no message anywhere. And **a key
pair belongs to one region**: an EU pair returns `401` against the US host. Match the host to
where you made the keys.

Skip step 2 entirely and the app still deploys and runs. You just get no LLM tracing, and the
script prints exactly what to put in the file.

In [ ]:
# ------------------------------------------------------- the few values code needs
# Everything else about this cluster is in the panel above -- read that, not this.
APP_IMAGE = "brainupgrade/frontdeskai:latest"
APP_PORT  = 8000            # what uvicorn listens on inside the container
APP_NAME  = "frontdeskai"   # every object in this lab is named after it

# The three Secrets this deployment reads. You never copy their values.
SECRETS = {
    "llm":      "{ns}-llm",                # published for you: gateway URL, model, your key
    "app":      "frontdeskai-secret",      # made at deploy: SECRET_KEY + first-login password
    "langfuse": "frontdeskai-langfuse",    # made at deploy from YOUR .env -- your own project
}

# The three quota numbers the self-checks below actually compare against.
QUOTA = {"limits.memory": "2Gi", "services": 4, "persistentvolumeclaims": 4}

# Tempo is shared by the whole cohort: your namespace may SEND to 4317, your
# sandbox may READ from 3200, which is what the last cell of this lab uses.
TEMPO_OTLP  = "http://tempo.monitoring.svc.cluster.local:4317"
TEMPO_QUERY = "http://tempo.monitoring.svc.cluster.local:3200"

NS   = APP_NS   or "your-namespace"      # placeholder keeps every cell runnable offline
HOST = APP_HOST or f"{NS}-app.example"

print("deploying :", APP_IMAGE)
print("namespace :", NS)
print("host      :", HOST)

## Section 1 &mdash; The five objects, and the five answers

A deployment of this app is five Kubernetes objects. You do not type them out &mdash; they are
written for you in the cell after next &mdash; but you do have to decide five things inside
them, and each answer is forced by a number in the panel above.

| Object | What it carries |
|---|---|
| **ConfigMap** | non-secret settings: which provider, which model, where spans go. Never a credential |
| **PersistentVolumeClaim** | 1Gi at `/shared` for SQLite and the vector store |
| **Deployment** | one replica, the image, both probes, requests and limits, and `envFrom` |
| **Service** | gives the pod a stable name inside the cluster, port 80 &rarr; 8000 |
| **Ingress** | claims your public hostname and sends it to the Service |

### The four decisions

| Decision | The fact that forces it | What to weigh |
|---|---|---|
| `update_strategy()` | one **ReadWriteOnce** volume holding SQLite | a `RollingUpdate` starts the replacement pod *before* stopping the old one. Both land on the same node, so both mount the same volume, and two processes writing one SQLite file is how a database gets corrupted. Quota bounds it too &mdash; add any second workload and the surge no longer fits |
| `service_type()` | `nodeports: 0`, `loadbalancers: 0` | an Ingress reaches a Service from *inside* the cluster, so it does not need either |
| `llm_provider()` | egress reaches `llm-serving` and nothing else | `groq`, `ollama` and `openrouter` each call a vendor over the internet. `litellm` points at an OpenAI-compatible gateway and reads its URL and key from the environment |
| `gateway_secret()` | the three Secrets listed above | which one holds the gateway credential |

### And one trap

`home_override()` looks like housekeeping and is not. The image bakes ChromaDB's embedding
model into `/opt/appcache` and points `HOME` there, because `chromadb` resolves its cache from
`Path.home()` and **downloads the model on first use** &mdash; which your namespace has no
egress to do. The app also mounts a PVC at `/shared`, which is a very tempting `HOME`.

Set `HOME` to `/shared` and the app crash-loops on `httpx.ConnectError` during startup
indexing, while every manifest is valid and both probes are configured correctly. It has
already broken this app once.

In [ ]:
# Five answers. The reasoning for each one is in the panel above -- none of these
# is a Python puzzle, and the mechanics are already written.

def update_strategy() -> dict:
    """One replica, one ReadWriteOnce volume, and SQLite on it."""
    return {"type": BLANK}              # "RollingUpdate" | "Recreate"


def service_type() -> str:
    """services.nodeports is 0, and so is services.loadbalancers."""
    return BLANK                        # "ClusterIP" | "NodePort" | "LoadBalancer"


def llm_provider() -> str:
    """Only the in-cluster gateway is reachable from this namespace."""
    return BLANK                        # "groq" | "ollama" | "openrouter" | "litellm"


def gateway_secret(ns: str) -> str:
    """Which Secret carries that credential. Return the key into SECRETS."""
    return SECRETS[BLANK].format(ns=ns) # "llm" | "langfuse" | "app"


def home_override() -> dict:
    """Extra env for HOME, if any. This is the trap."""
    return BLANK                        # {} | {"HOME": "/shared"}

In [ ]:
# The five objects, written for you. The five calls you filled in are the only
# things that vary.

def meta(name, ns):
    return {"name": name, "namespace": ns, "labels": {"app": APP_NAME}}


def build_configmap(ns):
    """Non-secret configuration. Nothing in here may be a credential."""
    env = {
        "LLM_PROVIDER":                llm_provider(),
        "LLM_MODEL":                   "qwen36-35b-a3b-lab",
        "LLM_FALLBACK_PROVIDER":       "",     # no second vendor is reachable from here
        "LLM_FALLBACK_MODEL":          "",     # empty model disables the fallback
        "SEED_DEMO_DATA":              "true",
        "SQLITE_DIR":                  "/shared/.sqlite",
        "OTEL_SERVICE_NAME":           f"{APP_NAME}-{ns}",
        "OTEL_EXPORTER_OTLP_ENDPOINT": TEMPO_OTLP,
        "LOG_LEVEL":                   "INFO",
        "ENV":                         "production",
    }
    env.update(home_override())
    return {"apiVersion": "v1", "kind": "ConfigMap",
            "metadata": meta(f"{APP_NAME}-config", ns), "data": env}


def build_deployment(ns):
    probe = lambda: {"httpGet": {"path": "/health", "port": APP_PORT}}
    return {
        "apiVersion": "apps/v1", "kind": "Deployment",
        "metadata": meta(APP_NAME, ns),
        "spec": {
            "replicas": 1,
            "strategy": update_strategy(),
            "selector": {"matchLabels": {"app": APP_NAME}},
            "template": {
                "metadata": {"labels": {"app": APP_NAME}},
                "spec": {
                    "securityContext": {"runAsNonRoot": True, "runAsUser": 1000, "fsGroup": 1000},
                    "containers": [{
                        "name": APP_NAME,
                        "image": APP_IMAGE,
                        "imagePullPolicy": "Always",
                        "ports": [{"containerPort": APP_PORT, "name": "http"}],
                        "envFrom": [
                            {"configMapRef": {"name": f"{APP_NAME}-config"}},
                            {"secretRef": {"name": gateway_secret(ns), "optional": True}},
                            {"secretRef": {"name": SECRETS["app"]}},
                        ],
                        "volumeMounts": [{"name": "data", "mountPath": "/shared"}],
                        "livenessProbe":  {**probe(), "initialDelaySeconds": 30, "periodSeconds": 30},
                        "readinessProbe": {**probe(), "initialDelaySeconds": 15, "periodSeconds": 10},
                        "resources": {
                            "requests": {"cpu": "200m", "memory": "256Mi"},
                            "limits":   {"cpu": "500m", "memory": "1Gi"},
                        },
                        "securityContext": {"allowPrivilegeEscalation": False,
                                            "capabilities": {"drop": ["ALL"]}},
                    }],
                    "volumes": [{"name": "data",
                                 "persistentVolumeClaim": {"claimName": f"{APP_NAME}-pvc"}}],
                },
            },
        },
    }


def build_service(ns):
    return {"apiVersion": "v1", "kind": "Service", "metadata": meta(APP_NAME, ns),
            "spec": {"type": service_type(), "selector": {"app": APP_NAME},
                     "ports": [{"name": "http", "port": 80, "targetPort": APP_PORT}]}}


def build_ingress(ns, host):
    return {"apiVersion": "networking.k8s.io/v1", "kind": "Ingress",
            "metadata": {**meta(APP_NAME, ns), "annotations": {
                "nginx.ingress.kubernetes.io/proxy-read-timeout": "300",
                "nginx.ingress.kubernetes.io/proxy-body-size": "16m"}},
            "spec": {"ingressClassName": "nginx", "rules": [{"host": host, "http": {"paths": [
                {"path": "/", "pathType": "Prefix",
                 "backend": {"service": {"name": APP_NAME, "port": {"number": 80}}}}]}}]}}


def build_pvc(ns):
    return {"apiVersion": "v1", "kind": "PersistentVolumeClaim",
            "metadata": meta(f"{APP_NAME}-pvc", ns),
            "spec": {"accessModes": ["ReadWriteOnce"],
                     "resources": {"requests": {"storage": "1Gi"}}}}


def all_manifests(ns=None, host=None):
    """Built lazily, so an unfilled blank prints [TODO] instead of crashing the cell."""
    ns, host = ns or NS, host or HOST
    return [build_configmap(ns), build_pvc(ns), build_deployment(ns),
            build_service(ns), build_ingress(ns, host)]

In [ ]:
# --- Self-check: the manifest set  (pure dicts -- no cluster, no model, no network)
def by_kind(kind):
    return next(m for m in all_manifests() if m["kind"] == kind)

def container():
    return by_kind("Deployment")["spec"]["template"]["spec"]["containers"][0]

def mem_to_mi(v):
    return int(v[:-2]) * 1024 if v.endswith("Gi") else int(v[:-2])

# the four decisions
check("Recreate, because two pods must not share one SQLite volume",
      lambda: by_kind("Deployment")["spec"]["strategy"]["type"] == "Recreate",
      "a RollingUpdate starts the new pod before stopping the old one, and both mount it")

check("ClusterIP, because the quota allows no NodePort",
      lambda: by_kind("Service")["spec"]["type"] == "ClusterIP"
              and not any("nodePort" in json.dumps(m) for m in all_manifests()))

check("the app is pointed at the in-cluster gateway",
      lambda: by_kind("ConfigMap")["data"]["LLM_PROVIDER"] == "litellm",
      "the other three providers need internet egress this namespace does not have")

check("the gateway credential comes from your own -llm Secret",
      lambda: any(r.get("secretRef", {}).get("name", "").endswith("-llm")
                  for r in container()["envFrom"]))

# the trap
check("HOME is left exactly as the image set it",
      lambda: "HOME" not in by_kind("ConfigMap")["data"],
      "moving HOME hides the baked embedding model, and there is no egress to re-fetch it")

# credentials
check("config and credentials arrive by reference, not by value",
      lambda: not container().get("env"),
      "envFrom only -- a literal in a manifest is a credential in git")

# the pod can be told whether it is healthy
check("both probes exist and ask the port the container listens on",
      lambda: container()["livenessProbe"]["httpGet"]["port"] == APP_PORT == 8000
              and container()["readinessProbe"]["httpGet"]["port"] == APP_PORT)

check("the memory limit does not exceed the whole namespace budget",
      lambda: mem_to_mi(container()["resources"]["limits"]["memory"])
              <= mem_to_mi(QUOTA["limits.memory"]))

# the route in
check("the Service targets the container port, and the Ingress targets the Service",
      lambda: by_kind("Service")["spec"]["ports"][0]["targetPort"] == APP_PORT
              and by_kind("Ingress")["spec"]["rules"][0]["http"]["paths"][0]
                  ["backend"]["service"]["port"]["number"] == 80)

check("the Ingress claims your own host",
      lambda: by_kind("Ingress")["spec"]["rules"][0]["host"] == HOST)

# telemetry
check("spans are exported, not silently dropped",
      lambda: by_kind("ConfigMap")["data"]["OTEL_EXPORTER_OTLP_ENDPOINT"] == TEMPO_OTLP,
      "an EMPTY endpoint disables the exporter; a WRONG one drops every span in silence")

check("your traces are findable among the whole cohort's",
      lambda: by_kind("ConfigMap")["data"]["OTEL_SERVICE_NAME"].endswith(NS),
      "one Tempo, 31 services -- the name has to carry your namespace")

check("every object lands in your namespace, and inside the quota",
      lambda: {m["metadata"]["namespace"] for m in all_manifests()} == {NS}
              and sum(1 for m in all_manifests() if m["kind"] == "Service") <= QUOTA["services"]
              and sum(1 for m in all_manifests()
                      if m["kind"] == "PersistentVolumeClaim") <= QUOTA["persistentvolumeclaims"])

## Section 2 &mdash; Put it on the cluster

You are not going to hand-apply the JSON you just linted. The repo ships
`scripts/deploy-spark.sh`, which takes **no arguments** &mdash; `APP_NAMESPACE` and `APP_HOST`
are already exported in your sandbox &mdash; and it applies the same five objects plus two
Secrets, one of them built from your `.env`.

So why lint the dicts at all? Because the script is somebody else's decision until you have
checked it. The next four cells, in order:

1. **clone** the app repo, or pull it if you already have it
2. **check your `.env`** &mdash; which of the three Langfuse names are set, never their values
3. **compare the repo's manifests against your own five answers**, and print where they differ
4. **run the script**, which creates the Secrets and applies the set

Step 3 is the one worth slowing down for. If the repo and your answer disagree, one of you is
wrong about this cluster, and finding out now is cheaper than finding out from a hung rollout.

These cells are marked **Run it for real**: they need your namespace, and they print what to do
instead when it is not set.

In [ ]:
# --- Run it for real: get the app's source -----------------------------------
import subprocess

APP_REPO_URL = "https://github.com/brainupgrade-in/aiagentic-comp-frontdeskai.git"
APP_REPO_DIR = os.path.join(HOME_WORK, "frontdeskai")

def sh(*args, cwd=None):
    p = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    return p.returncode, (p.stdout + p.stderr).strip()

def clone_app():
    if os.path.isdir(os.path.join(APP_REPO_DIR, ".git")):
        rc, out = sh("git", "pull", "--ff-only", cwd=APP_REPO_DIR)
        print("already cloned; git pull ->", out.splitlines()[-1] if out else "ok")
    else:
        rc, out = sh("git", "clone", "--depth", "20", APP_REPO_URL, APP_REPO_DIR)
        if rc:
            print("clone failed:", out[-400:]); return
        print("cloned into", APP_REPO_DIR)
    for f in ("scripts/deploy-spark.sh", ".env.example", "app/agents.py"):
        mark = "ok " if os.path.exists(os.path.join(APP_REPO_DIR, f)) else "MISSING "
        print(f"  {mark}{f}")

# Gated on APP_NAMESPACE like every other live cell. Without it you are not in a
# sandbox, and an offline verification run must not reach GitHub or write a clone.
if APP_NS:
    clone_app()
else:
    print("APP_NAMESPACE is unset, so this is not a sandbox -- skipping the clone.")
    print("In a sandbox terminal it is already exported.")

In [ ]:
# --- Run it for real: your own Langfuse keys, in the app's .env --------------
# This only reports WHICH names are set. It never prints a key.
ENV_FILE = os.path.join(APP_REPO_DIR, ".env")

def check_env():
    if not os.path.exists(ENV_FILE):
        print("No .env yet. In a terminal:")
        print(f"  cp {APP_REPO_DIR}/.env.example {ENV_FILE}")
        print(f"  nano {ENV_FILE}        # fill in the three LANGFUSE_ values")
        return
    seen = {}
    for line in open(ENV_FILE, encoding="utf-8", errors="replace"):
        line = line.strip()
        if line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        seen[k.strip()] = v.strip().strip("\"'")

    host = seen.get("LANGFUSE_HOST") or seen.get("LANGFUSE_BASE_URL") or ""
    for name in ("LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY"):
        print(f"  {'set    ' if seen.get(name) else 'MISSING'} {name}")
    print(f"  {'set    ' if host else 'MISSING'} LANGFUSE_HOST -> {host or '(none)'}")

    if seen.get("LANGFUSE_BASE_URL") and not seen.get("LANGFUSE_HOST"):
        print("\n  note: you used LANGFUSE_BASE_URL. The deploy accepts it, the app reads")
        print("        LANGFUSE_HOST, and the script translates. Either name works.")
    if host and not any(host.startswith(p) for p in ("http://", "https://")):
        print("\n  warning: the host needs its scheme -- https://<region>.cloud.langfuse.com")
    if all(seen.get(k) for k in ("LANGFUSE_PUBLIC_KEY", "LANGFUSE_SECRET_KEY")) and host:
        print("\n  Ready. The deploy will turn these into the frontdeskai-langfuse Secret.")
    else:
        print("\n  Without all three the app still deploys -- you just get no LLM tracing.")

check_env()

In [ ]:
# --- Run it for real: does the repo agree with the decisions you made? -------
# You predicted five things from your namespace's limits. The manifests the deploy
# script is about to apply are in the clone. Check them against your answers rather
# than taking anyone's word for it -- including this lab's.
import re

def verify_repo():
    m = os.path.join(APP_REPO_DIR, "scripts", "manifests", "spark")
    if not os.path.isdir(m):
        print("No clone yet -- run the clone cell above."); return
    read = lambda f: open(os.path.join(m, f), encoding="utf-8").read()
    dep, svc, cm = read("deployment.yaml"), read("service.yaml"), read("configmap.yaml")

    expected = [
        ("strategy",  update_strategy()["type"],
         re.search(r"strategy:\s*\n\s*type:\s*(\w+)", dep).group(1)),
        ("Service type", service_type(),
         re.search(r"type:\s*(\w+)", svc).group(1)),
        ("LLM provider", llm_provider(),
         re.search(r'LLM_PROVIDER:\s*"?(\w+)', cm).group(1)),
    ]
    for label, mine, theirs in expected:
        verdict = "agrees" if mine == theirs else "DIFFERS"
        print(f"  {verdict:8} {label:14} you: {mine!r}  repo: {theirs!r}")

    home_in_cm = bool(re.search(r"^\s{2}HOME:", cm, re.M))
    want_home  = bool(home_override())
    print(f"  {'agrees' if home_in_cm == want_home else 'DIFFERS':8} HOME override  "
          f"you: {want_home}  repo: {home_in_cm}")
    print(f"  {'agrees' if 'frontdeskai-langfuse' in dep else 'DIFFERS':8} "
          f"your Langfuse Secret is mounted, optional")

guard(verify_repo)

In [ ]:
# --- Run it for real: let the repo's own script deploy it --------------------
# One command. It creates the two Secrets -- including yours, from the .env -- and
# applies the five objects. Everything it does is in scripts/deploy-spark.sh, which
# is worth reading: it is the same five objects you just linted.
def kubectl(*args, **kw):
    return subprocess.run(("kubectl", "-n", APP_NS) + args,
                          capture_output=True, text=True, **kw)

def deploy():
    if not APP_NS or not APP_HOST:
        print("APP_NAMESPACE / APP_HOST are unset in this kernel, so there is nothing")
        print("to deploy into. In a sandbox terminal both are already exported.")
        return False
    script = os.path.join(APP_REPO_DIR, "scripts", "deploy-spark.sh")
    if not os.path.exists(script):
        print("No clone yet -- run the clone cell above."); return False
    p = subprocess.run(("bash", script), cwd=APP_REPO_DIR,
                       capture_output=True, text=True, timeout=900)
    print((p.stdout + p.stderr).strip()[-2500:])
    return p.returncode == 0

APPLIED = guard(deploy) or False

### Waiting, and asking the probe its own question

The first image pull is the slow part, so the rollout can take a couple of minutes.

Then the cell asks `/health` **from inside the running container**. The obvious way would be to
`kubectl run` a small curl pod, and you have just enough quota for one &mdash; but a debugging pod
is still a pod against your budget, and another image to pull. `exec` borrows the container you
have already paid for, and this image ships Python. On the old 1Gi quota it was not a choice at
all: the debug pod was refused outright.

In [ ]:
# --- Run it for real: wait for it, then ask the readiness probe ------------
def wait_for_it():
    if not APPLIED:
        print("Nothing was applied, so there is nothing to wait for.")
        return
    print("waiting for the rollout (the first image pull is the slow part)...")
    out = kubectl("rollout", "status", "deployment/frontdeskai", "--timeout=300s")
    print(out.stdout.strip() or out.stderr.strip())

    pods = kubectl("get", "pods", "-l", "app=frontdeskai", "--no-headers")
    print(pods.stdout.strip())

    hit = kubectl("exec", f"deploy/{APP_NAME}", "--", "python3", "-c",
                  "import urllib.request as u;"
                  f"print(u.urlopen('http://127.0.0.1:{APP_PORT}/health').status)")
    print("GET /health ->", (hit.stdout.strip() or hit.stderr.strip()[:200]))
    print("\nyour app:", "https://" + APP_HOST if APP_HOST else "(APP_HOST unset)")

guard(wait_for_it)

### Asking the running service a real question

This is the whole course in one HTTP request: supervisor &rarr; RAG &rarr; a domain worker with
tools &rarr; escalation check &rarr; QA gate &rarr; a grounded answer. Watch the audit trail it
prints &mdash; that is the same trail you have been reading all week, now coming out of a
service instead of a notebook.

⚠️ **It asks the pod directly rather than your public hostname, and that is deliberate.**
Cloudflare sits in front of that host and gives up at about 100 seconds with a `524`; an agent
doing three tool-calling turns can exceed that whenever the gateway is busy, and the `524` looks
exactly like a broken app while the pod is still working. It also answers the default Python
user agent with `403 error code: 1010`. **Open the hostname in a browser** to see the real
thing, and let this cell use `kubectl exec`.

In [ ]:
# --- Run it for real: ask the running service a question -------------------
def talk_to_it():
    if not APPLIED:
        print("The app is not deployed from this kernel, so there is nothing to ask.")
        return
    ask_py = (
        "import json,urllib.request,urllib.parse,http.cookiejar;"
        "j=http.cookiejar.CookieJar();"
        "o=urllib.request.build_opener(urllib.request.HTTPCookieProcessor(j));"
        "p=lambda u,d: o.open('http://127.0.0.1:8000'+u,"
        " data=urllib.parse.urlencode(d).encode(), timeout=600);"
        "p('/login', {'email':'rajesh.kumar@unigps.in','password':'brainupgrade'});"
        "r=json.loads(p('/chat/send', {'message':'What is my current leave balance?'}).read());"
        "print(json.dumps({'category':r['category'],'audit':r['audit'],"
        "'response':r['response'][:400]}))"
    )
    out = kubectl("exec", f"deploy/{APP_NAME}", "--", "python3", "-c", ask_py)
    if out.returncode != 0:
        print("the request did not complete:", (out.stderr or out.stdout).strip()[:300])
        return
    try:
        answer = json.loads(out.stdout.strip().splitlines()[-1])
    except Exception:
        print(out.stdout.strip()[:400])
        return
    print("category:", answer["category"])
    for line in answer["audit"]:
        print("  ", line)
    print()
    print(answer["response"])

guard(talk_to_it)

### Reading your own trace back out

The request you just made emitted spans. This asks Tempo what it actually **received**.

That distinction is the point: *&ldquo;the exporter is configured&rdquo;* is not evidence.
`BatchSpanProcessor` swallows export failures, so a wrong endpoint looks exactly like a healthy
one from inside the app &mdash; no error, no warning, no spans. The only proof is reading them
back.

Spans are batched, so give it about ten seconds after a request. If the newest trace looks
partial, that is why: a trace's root span lands last.

In [ ]:
# --- Run it for real: read your own trace back out of Tempo ----------------
def read_my_traces():
    import urllib.request, urllib.parse
    if not APP_NS:
        print("APP_NAMESPACE is unset in this kernel, so there is no service to look up.")
        return
    svc = f"{APP_NAME}-{APP_NS}"

    def get(path):
        return json.load(urllib.request.urlopen(TEMPO_QUERY + path, timeout=30))

    try:
        q = urllib.parse.quote(f"service.name={svc}")
        found = get(f"/api/search?tags={q}&limit=5").get("traces", [])
    except Exception as exc:
        print(f"could not reach Tempo ({type(exc).__name__}). Spans are batched, so give")
        print("it about ten seconds after a request and run this cell again.")
        return

    if not found:
        print(f"Tempo has no traces for {svc} yet -- spans are batched. Wait ~10s, re-run.")
        return

    print(f"{len(found)} trace(s) for {svc}")
    complete = [t for t in found if t.get("rootTraceName")] or found
    newest = max(complete, key=lambda t: int(t.get("startTimeUnixNano", 0)))
    print(f"  {newest.get('rootTraceName')}  {newest.get('durationMs')} ms  {newest['traceID']}")
    print()

    rows = []
    for b in get(f"/api/traces/{newest['traceID']}").get("batches", []):
        for ss in b.get("scopeSpans", []):
            for sp in ss.get("spans", []):
                ms = (int(sp["endTimeUnixNano"]) - int(sp["startTimeUnixNano"])) / 1e6
                attrs = {a["key"]: list(a["value"].values())[0]
                         for a in sp.get("attributes", [])}
                rows.append((ms, sp["name"], attrs))
    rows.sort(reverse=True)
    print("  slowest spans:")
    for ms, name, attrs in rows[:6]:
        extra = attrs.get("llm.tokens") or attrs.get("chat.category") or ""
        print(f"    {name:30} {ms:9.1f} ms  {extra}")
    print()
    print("  Which step dominates? That is what a trace answers and a latency metric")
    print(f"  does not. In Grafana: Explore -> Tempo -> service.name = {svc}")

guard(read_my_traces)

In [ ]:
score()

## Your turn

1. **Break it deliberately.** Set `home_override()` to `{"HOME": "/shared"}`, redeploy, and read
   the crash. That is the exact failure this app hit on its first deployment here: chromadb
   resolves its cache from `Path.home()`, finds nothing, tries to download the model, and dies
   during startup indexing with a connection error &mdash; while every manifest is valid and
   every probe is configured. **A healthy manifest is not a healthy app.**
2. **Make the rollout hang.** Deploy anything else with a 1Gi limit into your namespace, then
   set `update_strategy()` to `{"type": "RollingUpdate"}` and apply. The surge pod no longer
   fits, the old one keeps serving, and `rollout status` waits until it times out &mdash; with
   nothing you would notice from the outside. Find the message that *does* say what happened
   &mdash; `kubectl -n $APP_NAMESPACE describe rs` &mdash; and decide where in a pipeline you
   would surface it. (This is not hypothetical: a namespace with an nginx and a postgres already
   in it could not deploy this app at all until the quota was raised on 2026-09-11.)
3. **Pin the image.** Replace `:latest` with the digest of the image you just deployed
   (`kubectl get pod -o jsonpath='{..imageID}'`), and say what that buys and what it costs.
4. Which of the eighteen checks above would have caught **only** a mistake, and which encode a
   fact about *this* cluster that would be wrong somewhere else? Those are two different kinds of
   rule and only one of them travels.

**What you take from Module 9:** the constraints of the environment are part of the design, the
manifest is where that design is written down, and a set of predicates over it is a review that
runs every time instead of a checklist someone remembers.

That is the last lab. The capstone puts all nine modules behind one endpoint.